# 🛰️ Example: Working with the ComSat Environment

This notebook demonstrates the usage of the **ComSat** (communication satellite) environment from the TensorAeroSpace library.

## 📋 Contents
1. Import the required libraries
2. Configure simulation parameters
3. Create and initialize the environment
4. Execute a simulation step
5. Analyze the results

---

## 📚 Importing Libraries

Loading all necessary modules for working with the communication satellite environment:

In [1]:
# Core libraries for working with environments and numerical computations
import gymnasium as gym 
import numpy as np
from tqdm import tqdm

# Specialized TensorAeroSpace modules
from tensoraerospace.envs import ComSatEnv
from tensoraerospace.utils import generate_time_period, convert_tp_to_sec_tp
from tensoraerospace.signals.standart import unit_step

## ⚙️ Simulation Parameters Setup

Defining the main parameters for modeling satellite dynamics:

In [2]:
# Discretization and time parameters
dt = 0.01  # Discretization step (seconds)
tp = generate_time_period(tn=20, dt=dt)  # Time period (20 seconds)
tps = convert_tp_to_sec_tp(tp, dt=dt)    # Time period conversion
number_time_steps = len(tp)              # Total number of time steps

# Generating the reference signal (unit step)
reference_signals = np.reshape(
    unit_step(degree=5, tp=tp, time_step=10, output_rad=True), 
    [1, -1]
)  # Specified control signal

## 🚀 Creating the ComSat Environment

Initializing the communication satellite environment with the specified parameters:

In [3]:
# Creating the communication satellite environment
env = gym.make('ComSat-v0',
               number_time_steps=number_time_steps,  # Number of time steps
               initial_state=[[0],[0],[0]],          # Initial system state
               output_space=None,                    # Output space (automatic)
               reference_signal=reference_signals)   # Reference signal

# Reset environment to initial state
env.reset()

(array([[0.],
        [0.],
        [0.]], dtype=float32),
 {})

## 🎯 Executing a Simulation Step

Applying a control action and obtaining the system response:

In [4]:
# Executing one simulation step with a control signal
observation, reward, terminated, truncated, info = env.step(np.array([[1]]))

/home/mr8bit/.pyenv/versions/3.12.7/lib/python3.12/site-packages/gymnasium/utils/passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `step()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
/home/mr8bit/.pyenv/versions/3.12.7/lib/python3.12/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


## 📊 Analyzing the Results

Examining the obtained system state data:

In [5]:
# Viewing control action history
env.unwrapped.model.store_input

array([[1., 0., 0., ..., 0., 0., 0.]], shape=(1, 2002))

In [6]:
# Reference signal value at the second time step
env.unwrapped.reference_signal[0][1]

np.float64(0.0)

In [7]:
# Reward received for the performed action
reward

-5.865144333743929e-06

## Visualization: Full Simulation with Plots

Run the full simulation loop and visualize the ComSat orbital dynamics.
We re-create the environment, step through all time steps with a simple
proportional controller, and plot the key orbital parameters.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    'figure.facecolor': '#0e1117',
    'axes.facecolor': '#1a1d23',
    'axes.edgecolor': '#3d4253',
    'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#2d333b',
    'legend.facecolor': '#1a1d23',
    'legend.edgecolor': '#3d4253',
})

# -- Re-create and run the full simulation --
env_viz = gym.make(
    'ComSat-v0',
    number_time_steps=number_time_steps,
    initial_state=[[0], [0], [0]],
    output_space=None,
    reference_signal=reference_signals,
)
obs, _ = env_viz.reset()

rewards = []
for i in range(number_time_steps - 1):
    # Simple proportional controller on theta_dot tracking error
    ref_val = env_viz.unwrapped.reference_signal[0][min(i, len(env_viz.unwrapped.reference_signal[0]) - 1)]
    current_theta_dot = obs[2, 0] if obs.ndim > 1 else obs[2]
    error = ref_val - current_theta_dot
    action = np.clip(np.array([[error * 5.0]]), -25, 25)
    obs, reward, terminated, truncated, info = env_viz.step(action)
    rewards.append(reward)
    if terminated or truncated:
        break

model = env_viz.unwrapped.model
n = len(rewards)
time_axis = np.arange(n) * dt

theta_dot = model.get_state('theta_dot')[:n]
rho = model.get_state('rho')[:n]
u2 = model.get_control('u2')[:n]
ref = env_viz.unwrapped.reference_signal[0][:n]

# -- Plot --
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('ComSat -- Communication Satellite Orbital Dynamics',
             fontsize=15, fontweight='bold', color='#58a6ff')

# 1 - theta_dot tracking
ax = axes[0, 0]
ax.plot(time_axis, np.rad2deg(ref), '--', color='#f97583', linewidth=1.5, label='Reference')
ax.plot(time_axis, np.rad2deg(theta_dot), color='#79c0ff', linewidth=1.2, label='Actual')
ax.set_title('Angular Velocity Tracking', color='#58a6ff')
ax.set_xlabel('Time (s)')
ax.set_ylabel(r'$\dot{\theta}$ (deg/s)')
ax.legend()
ax.grid(True, alpha=0.3)

# 2 - radial position
ax = axes[0, 1]
ax.plot(time_axis, rho, color='#7ee787', linewidth=1.2)
ax.set_title('Radial Position', color='#58a6ff')
ax.set_xlabel('Time (s)')
ax.set_ylabel(r'$\rho$ (km)')
ax.grid(True, alpha=0.3)

# 3 - control input
ax = axes[1, 0]
ax.plot(time_axis, u2, color='#d2a8ff', linewidth=1.0)
ax.set_title('Control Input -- Tangential Thrust', color='#58a6ff')
ax.set_xlabel('Time (s)')
ax.set_ylabel(r'$u_2$ (N)')
ax.grid(True, alpha=0.3)

# 4 - reward
ax = axes[1, 1]
ax.plot(time_axis, rewards, color='#ffa657', linewidth=0.8)
ax.set_title('Reward over Time', color='#58a6ff')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Reward')
ax.grid(True, alpha=0.3)

fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
env_viz.close()

## 🎉 Conclusion

In this example we successfully:

- ✅ Configured the communication satellite simulation parameters
- ✅ Created the ComSat environment with the specified characteristics
- ✅ Executed a simulation step with a control action
- ✅ Analyzed the system operation results

The obtained reward `5.86126179e-06` indicates the control quality at this step.

---

💡 **Tip**: For a more detailed analysis, it is recommended to perform several simulation steps and plot the system state changes over time.